<a href="https://colab.research.google.com/github/eshikajindal24/UCS547_Accelerated_data_science/blob/main/UCS547_102303954_Assign3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT 3**

In [ ]:
!nvidia-smi

Sun Feb 15 17:07:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

1. Write a CUDA C/C++ program to perform element-wise addition of two
vectors.
C[i]=A[i]+B[i]
Given: Vector size: N = 1024

In [ ]:
%%writefile addvector.cu
#include<stdio.h>
#include <cuda_runtime.h>
#define N 16

__global__ void add(int *a, int *b, int *c){
int index= blockIdx.x * blockDim.x + threadIdx.x;
c[index] = a[index] + b[index];
};
int main(void)
{
int *a, *b, *c;
int *d_a, *d_b, *d_c;
int size = N * sizeof(int);
a = (int *)malloc(size);
b = (int *)malloc(size);
c = (int *)malloc(size);

printf("\nValues for a\n");
for (int i = 0; i < N; i++) {
a[i] = i * 1;
printf("%d  ",a[i]);
}

printf("\nValues for b\n");
for (int i = 0; i < N; i++) {
b[i] = i * 2;
printf("%d  ",b[i]);
}

cudaMalloc((void **)&d_a, size);
cudaMalloc((void **)&d_b, size);
cudaMalloc((void **)&d_c, size);

cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

int threads = 8;
int blocks = (N + threads - 1) / threads;

add<<<blocks,threads>>>(d_a, d_b, d_c);

cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
printf("\nArray values of C:\n");
for(int i=0; i<N; i++)
printf("%d ", c[i]);

cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
return 0;
}

Overwriting addvector.cu


In [ ]:
!nvcc -arch=sm_75 addvector.cu -o add

In [ ]:
! ./add


Values for a
0  1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  
Values for b
0  2  4  6  8  10  12  14  16  18  20  22  24  26  28  30  
Array values of C:
0 3 6 9 12 15 18 21 24 27 30 33 36 39 42 45 

2. Perform the same vector addition as in Q1 using Thrust library only.

In [ ]:
%%writefile taddvector.cu
#include <thrust/device_vector.h>
#include <thrust/transform.h>
#include <iostream>

#define N 1024

int main() {
    thrust::device_vector<int> A(N,1);
    thrust::device_vector<int> B(N,2);
    thrust::device_vector<int> C(N);

    thrust::transform(A.begin(), A.end(),
                      B.begin(),
                      C.begin(),
                      thrust::plus<int>());

    std::cout << "C[0] = " << C[0] << std::endl;
}


Overwriting taddvector.cu


In [ ]:
!nvcc -arch=sm_75 taddvector.cu -o tadd

In [ ]:
! ./tadd

C[0] = 3


3. Compute the dot product of two vectors of size, N =1024: Result=∑A[i]×B[i]
using Thrust and compare its performance with that on CPU.

In [47]:
%%writefile q3.cu
#include <thrust/device_vector.h>
#include <thrust/inner_product.h>
#include <iostream>

#define N 1024

int main() {
    thrust::device_vector<int> A(N,1);
    thrust::device_vector<int> B(N,2);

    int result = thrust::inner_product(A.begin(), A.end(),
                                       B.begin(), 0);

    std::cout << "Dot Product = " << result << std::endl;
}

Writing q3.cu


In [48]:
!nvcc q3.cu -o xyz

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [49]:
! ./xyz

Dot Product = 2048


4. Write a CUDA kernel for matrix multiplication: C=A×B where Matrix size is 16
X 16. Explain why matrix multiplication needs more computation than
addition (as in Q1).

In [53]:
%%writefile matmul.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 16

__global__ void matMul(int A[N][N], int B[N][N], int C[N][N]) {
    int row = threadIdx.y;
    int col = threadIdx.x;
    int sum = 0;

    for (int k = 0; k < N; k++)
        sum += A[row][k] * B[k][col];

    C[row][col] = sum;
}

int main() {
    int A[N][N], B[N][N], C[N][N];
    int (*d_A)[N], (*d_B)[N], (*d_C)[N];

    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++) {
            A[i][j] = 1;
            B[i][j] = 1;
        }

    cudaMalloc(&d_A, sizeof(A));
    cudaMalloc(&d_B, sizeof(B));
    cudaMalloc(&d_C, sizeof(C));

    cudaMemcpy(d_A, A, sizeof(A), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, sizeof(B), cudaMemcpyHostToDevice);

    dim3 threads(N, N);
    matMul<<<1, threads>>>(d_A, d_B, d_C);

    cudaMemcpy(C, d_C, sizeof(C), cudaMemcpyDeviceToHost);

    printf("C[0][0] = %d\n", C[0][0]);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}

Overwriting matmul.cu


In [54]:
!nvcc matmul.cu -o matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [55]:
! ./matmul

C[0][0] = 16


Vector addition → 1 operation per element

Matrix multiplication → N multiplications + (N−1) additions per element

Complexity:

Vector add: O(N)

Matrix multiply: O(N³)

5. For vector addition of size 5,000,000, implement and compare:
• CPU sequential C/C++ program
• CUDA kernel implementation
• Thrust implementation
• RAPIDS implementation
Measure execution time and compare complexity for each approach and
present results in a table. Plot comparison graph.

CPU

In [67]:
import numpy as np
import time

N = 5000000

A = np.arange(N)
B = np.arange(N)

start = time.time()
C = A + B
end = time.time()

print("CPU Time:", end - start)

CPU Time: 0.01590871810913086


CUDA

In [68]:
%%writefile cuda_add.cu
#include <stdio.h>
#include <cuda.h>

#define N 5000000

__global__ void cuda_add(int *A, int *B, int *C) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N)
        C[i] = A[i] + B[i];
}

int main() {

    int *h_A, *h_B, *h_C;
    int *d_A, *d_B, *d_C;

    size_t size = N * sizeof(int);

    h_A = (int*)malloc(size);
    h_B = (int*)malloc(size);
    h_C = (int*)malloc(size);

    for(int i=0;i<N;i++){
        h_A[i] = i;
        h_B[i] = i;
    }

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    float start = clock();

    cuda_add<<<(N+255)/256,256>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();

    float end = clock();

    printf("CUDA Time: %f\n",(end-start)/CLOCKS_PER_SEC);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
}


Writing cuda_add.cu


In [69]:
!nvcc cuda_add.cu -o cuda_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cuda_add.cu(14): warning #550-D: variable "h_C" was set but never used
      int *h_A, *h_B, *h_C;
                       ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [70]:
! ./cuda_add

CUDA Time: 0.012681


Thrust

In [71]:
%%writefile thrust_add.cu
#include <thrust/device_vector.h>
#include <thrust/transform.h>
#include <iostream>
#include <time.h>

#define N 5000000

int main() {

    thrust::device_vector<int> A(N);
    thrust::device_vector<int> B(N);
    thrust::device_vector<int> C(N);

    for(int i=0;i<N;i++){
        A[i] = i;
        B[i] = i;
    }

    float start = clock();

    thrust::transform(A.begin(), A.end(),
                      B.begin(),
                      C.begin(),
                      thrust::plus<int>());

    cudaDeviceSynchronize();

    float end = clock();

    std::cout << "Thrust Time: "
              << (end-start)/CLOCKS_PER_SEC
              << std::endl;
}

Writing thrust_add.cu


In [72]:
!nvcc thrust_add.cu -o thrust_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [73]:
!./thrust_add

Thrust Time: 0.00036


Rapids

In [74]:
import cupy as cp
import time

N = 5000000
A = cp.arange(N)
B = cp.arange(N)

start = time.time()
C = A + B
cp.cuda.Stream.null.synchronize()
print("RAPIDS Time:", time.time()-start)

RAPIDS Time: 0.10919022560119629


6. Write a CUDA C++ program using the Thrust library to compute the sum of
all elements in a vector stored on the GPU. The vector is of size 10 and it
should be initialized with values 1,…..10.

In [61]:
%%writefile ques6.cu
#include <thrust/device_vector.h>
#include <thrust/reduce.h>
#include <iostream>

int main() {
    thrust::device_vector<int> arr(10);

    for(int i=0;i<10;i++)
        arr[i] = i+1;

    int sum = thrust::reduce(arr.begin(), arr.end());

    std::cout << "Sum = " << sum << std::endl;
}



Overwriting ques6.cu


In [62]:
!nvcc ques6.cu -o ques6

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [63]:
! ./ques6

Sum = 55


7. Write a CUDA C++ program using Thrust to sort (ascending) a vector of
integers on the GPU. Consider vector size 8 with following values: 7, 2, 9, 1,
5, 3, 8, 4. Print the vector before and afer sorting.

In [64]:
%%writefile sort.cu
#include <thrust/device_vector.h>
#include <thrust/sort.h>
#include <iostream>

int main() {
    int arr[8] = {7,2,9,1,5,3,8,4};

    thrust::device_vector<int> nums(arr, arr+8);

    std::cout << "Before Sorting:\n";
    for(int i=0;i<8;i++)
        std::cout << nums[i] << " ";

    thrust::sort(nums.begin(), nums.end());

    std::cout << "\nAfter Sorting:\n";
    for(int i=0;i<8;i++)
        std::cout << nums[i] << " ";
}

Writing sort.cu


In [65]:
!nvcc sort.cu -o sort

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [66]:
! ./sort

Before Sorting:
7 2 9 1 5 3 8 4 
After Sorting:
1 2 3 4 5 7 8 9 